# 00_config — Shared Config & Helpers

`%run` by every other notebook. Centralizes the storage connection string, 
source list, table/volume names, and small logging/counting helpers.

**This is the connection-string variant of the pipeline.** Bronze reads 
files landed in ADLS using the plain Python `azure-storage-blob` SDK and a 
storage account connection string — not Auto Loader over an `abfss://` 
path, and not a Unity Catalog External Location. That's a deliberate 
choice, not a step backward: Databricks Serverless blocks 
`spark.conf.set("fs.azure.account.key...")` outright (`CONFIG_NOT_AVAILABLE`), 
which is what Auto Loader over `abfss://` would need without an External 
Location, and the External Location route needs grants (`READ FILES`, 
`WRITE FILES`, `CREATE MANAGED STORAGE`) that may not be available to you 
on a shared/class storage account. The Blob SDK never touches Spark's 
filesystem config at all -- it's a plain HTTPS call, same mechanism the 
live-fetch version (`py_work/`) already uses successfully against public 
APIs on serverless -- so it works regardless of either restriction.

**Note on architecture:** Bronze/Silver/Gold are **Unity Catalog managed 
tables** — `fomoless.bronze.articles_raw`, `fomoless.silver.articles`, 
`fomoless.silver.articles_rejected`, `fomoless.gold.articles`. `01_setup` 
creates the schemas (assumes the `fomoless` catalog itself already exists — 
see the note in `01_setup` if it doesn't yet).

**Bronze is a plain batch step, not a stream** — since there's no Auto 
Loader checkpoint tracking "what's already been read", incrementality 
comes from checking each landed file's blob name against what's already in 
`bronze.articles_raw` (`source_blob_name`) before downloading it, so a 
re-run only pulls genuinely new files. **Silver stays incremental** via its 
own Structured Streaming checkpoint, stored in a small Unity Catalog 
**managed Volume** (`VOLUME_PATH`, created by `01_setup`) — Databricks-managed 
storage under the catalog you already own, so it needs no extra grants 
either. That Volume also holds nothing else; dedup-merge audit logs go 
straight to ADLS via the same Blob SDK connection as bronze.

**New in this revision:** `CANONICAL_SOURCE_NAMES` (CHK-4 enum gate), 
`SOURCES_WITHOUT_NATIVE_SUMMARY` (CHK-5 conditional-completeness exception 
list), and `CLOCK_SKEW_LOG_PREFIX` (CHK-7 audit trail) — all consumed by 
`03_silver_transform`.

---
**Gold/Mongo revision:** added `ensure_cdf_enabled` (moved here from
`03_silver_transform` since `04_gold_aggregation` now needs the identical
logic — shared, not duplicated), `TREND_DECAY_HALF_LIFE_DAYS` (TR-11) and
the `CROSS_SOURCE_*` / `ENGAGEMENT_*` / `MOMENTUM_MULTIPLIER_*` constants
(TR-13, revised — see below), new `gold`/`mongo_sink` checkpoint paths,
and `MONGO_URI` (never hardcoded).

**TR-13 REVISED (this pass):** the old flat `MOMENTUM_MULTIPLIERS` lookup
only ever had one confirmed value (`BBC`/`The Guardian` → `3.0`), with
every other source silently defaulting to `1.0` — which is exactly what
let BBC dominate the trend rankings regardless of whether any individual
BBC article was actually resonating. It's been replaced with two
data-driven signals computed in `04_gold_aggregation`'s
`with_momentum_signals`: how many *other* sources are covering something
with an overlapping tag (`cross_source_score`), and how this article's
engagement ranks against its *own* source's recent history
(`engagement_score`). Both are derived from data already in the pipeline —
no new external dependency, and no business input required to be
structurally complete. See that function's docstring for the full
reasoning; the constants below only tune the computation.

## 1. Serverless "spark not initialized" workaround

On some Databricks Serverless sessions, the notebook-provided `spark` 
global isn't ready the first time a cell references it, raising 
`SystemError: Internal error: spark should be initialized with the first 
notebook command.` Explicitly building a session via `databricks.connect` 
fixes it and is a no-op anywhere `spark` is already fine.

In [0]:
import os
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import pandas as pd

In [0]:
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()

## 2. Storage connection string

From Azure Portal → your storage account → **Access keys** → Connection 
string (key 1 or key 2, either works). Settable via a Databricks widget 
(`connection_string`) or the `AZURE_STORAGE_CONNECTION_STRING` environment 
variable -- never hardcode it directly in a cell.

In [0]:
def get_widget_or_env(widget_name: str, env_var_name: str, label: str, default: str = ""):
    """Prefer a Databricks widget (settable by hand, or via a job/ADF base
    parameter), fall back to an environment variable, fall back to `default`
    for a local/interactive run where neither was set."""
    try:
        dbutils.widgets.text(widget_name, "", label)
        from_widget = dbutils.widgets.get(widget_name)
        if from_widget:
            return from_widget
    except NameError:
        pass
    return os.environ.get(env_var_name) or default


CONNECTION_STRING = get_widget_or_env(
    "connection_string", "AZURE_STORAGE_CONNECTION_STRING", "Azure Storage Connection String"
)
if not CONNECTION_STRING:
    raise EnvironmentError(
        "No Azure Storage connection string found. Supply it via the Databricks notebook widget "
        "'connection_string', or set AZURE_STORAGE_CONNECTION_STRING as an environment variable."
    )

CONTAINER_NAME = "fomoless"
RAW_PREFIX = "raw"
MERGE_LOG_PREFIX = "logs/dedup_merges"
GMAIL_APP_PASSWORD = "aqgrzyvbthedpocm"
# CHK-7 audit trail: rows whose Ingested_at is earlier than their
# Published_date get flagged (not rejected -- see 03_silver_transform) and
# also logged here for manual review, mirroring MERGE_LOG_PREFIX's pattern.
CLOCK_SKEW_LOG_PREFIX = "logs/clock_skew_flags"

## 3. Sources & schema

In [0]:
# folder = the raw/<folder>/ prefix in ADLS; kind = how 03_silver_transform parses raw_content
# (silver needs `kind` to know how to parse `raw_content`).
SOURCES = [
    {"folder": "devto", "kind": "json"},
    {"folder": "medium", "kind": "rss"},
    {"folder": "towardsdatascience", "kind": "rss"},
    {"folder": "bbc", "kind": "rss"},
    {"folder": "hackernews", "kind": "json"},
    {"folder": "stackoverflow_blog", "kind": "rss"},
    {"folder": "stackexchange", "kind": "json"},
    {"folder": "kdnuggets", "kind": "rss"},
    {"folder": "guardian", "kind": "json"},
]

# Silver's harmonized schema. NEW in this revision: raw_upvotes and
# raw_comments -- added because 04_gold_aggregation's TR-12 Engagement
# metric needs a raw engagement number that was never previously carried
# past silver's mappers. Nullable; only the three mappers with a native
# equivalent (dev.to's public_reactions_count/comments_count, Hacker
# News's points/num_comments, Stack Exchange's score/answer_count) populate
# them -- deliberately collapsed into ONE generic upvotes/comments pair
# rather than six per-source-named columns, since TR-12 only ever needs
# their sum, never the per-source distinction. Everyone else leaves them
# None. Dq_flags (CHK-5/CHK-7) is still deliberately NOT listed here -- see
# the note further down.
TARGET_COLUMNS = [
    "Article_ID", "Title", "Source_name", "Author", "Published_date",
    "Url", "Tag_1", "Tag_2", "Tag_3", "Tag_4", "Tag_5", "Tag_6",
    "Has_tag", "Summary", "Ingested_at", "Cleaned_at", "canonical_url",
    "raw_upvotes", "raw_comments",
]

# Dq_flags (CHK-5/CHK-7) is deliberately NOT listed above -- it's computed
# after validation, not at parse time, and is added to the validated
# DataFrame directly in 03_silver_transform with schema auto-merge handling
# the table evolution.

# CHK-4: the enum of valid Source_name values. NOTE -- the input-checks
# sheet this project was built from lists 8 sources (SRC-1..SRC-8); the
# actual pipeline has grown to 9 (Towards Data Science was split out of
# Medium). This list reflects the REAL 9 canonical names the mappers below
# actually assign, not the stale 8-source sheet -- enforcing the sheet's
# literal 8-item list would incorrectly reject every genuine Towards Data
# Science article.
CANONICAL_SOURCE_NAMES = {
    "dev.to", "Medium", "Towards Data Science", "BBC", "Hacker News",
    "Stack Overflow Blog", "Stack Exchange", "KDnuggets", "The Guardian",
}

# CHK-5: sources that structurally never provide a native summary (their
# mappers always pass summary=None -- see map_hackernews / map_stackexchange
# in 03_silver_transform). A missing Summary is only suspicious -- and only
# flagged -- for sources NOT in this set.
SOURCES_WITHOUT_NATIVE_SUMMARY = {"Hacker News", "Stack Exchange"}

HTML_SUMMARY_SOURCES = {"medium", "towardsdatascience", "guardian"}
MEDIUM_SOURCES = {"Medium", "Towards Data Science"}
TRUE_VALUES = {"true", "1", "yes", "y"}
FALSE_VALUES = {"false", "0", "no", "n"}
RIYADH_TZ = ZoneInfo("Asia/Riyadh")

# Minimum row count each layer must clear for 05_data_quality to pass.
DQ_MIN_ROWS = 1

## 4. Catalog, schemas, tables & volume

In [0]:
CATALOG_NAME = "fomoless"
SCHEMAS = {"bronze": "bronze", "silver": "silver", "gold": "gold"}

TABLES = {
    "bronze": f"{CATALOG_NAME}.{SCHEMAS['bronze']}.articles_raw",
    "silver": f"{CATALOG_NAME}.{SCHEMAS['silver']}.articles",
    "silver_rejected": f"{CATALOG_NAME}.{SCHEMAS['silver']}.articles_rejected",
    "gold": f"{CATALOG_NAME}.{SCHEMAS['gold']}.articles",
}

# A managed Volume, created by 01_setup, for streaming checkpoints --
# Databricks-managed storage under a catalog/schema you already own, so it
# needs no External Location grants.
VOLUME_NAME = "pipeline_internal"
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMAS['silver']}/{VOLUME_NAME}"
CHECKPOINTS = {
    "silver": f"{VOLUME_PATH}/checkpoints/silver_stream",
    # New: 04_gold_aggregation and 06_load_mongodb are both now CDF-driven
    # streams (see their own notebooks), so each needs its own checkpoint,
    # same pattern as silver's.
    "gold": f"{VOLUME_PATH}/checkpoints/gold_stream",
    "mongo_sink": f"{VOLUME_PATH}/checkpoints/mongo_sink_stream",
}

## 5. Gold trend-scoring constants (TR-11, TR-13)

In [0]:
TREND_DECAY_HALF_LIFE_DAYS = 7.0

# TR-13 (REVISED): the old MOMENTUM_MULTIPLIERS lookup only ever had one
# confirmed value (BBC/The Guardian -> 3.0), with every other source
# silently defaulting to 1.0 -- which is exactly what let BBC dominate the
# rankings regardless of whether any individual BBC article was actually
# resonating. Replaced with two data-driven signals computed in
# 04_gold_aggregation's with_momentum_signals: how many OTHER sources are
# covering something with an overlapping tag (cross_source_score), and how
# this article's engagement ranks against its OWN source's recent history
# (engagement_score). These constants tune that computation.
CROSS_SOURCE_WINDOW_DAYS = 5       # how many days apart two articles can be and still count as "same story window"
CROSS_SOURCE_MAX_SOURCES = 3       # cross_source_score hits its max (1.0) once this many OTHER distinct sources overlap
ENGAGEMENT_LOOKBACK_DAYS = 30      # how far back to look when ranking an article's Engagement against its own source's recent history
MOMENTUM_MULTIPLIER_BASE = 1.0
MOMENTUM_MULTIPLIER_CAP = 2.0
CROSS_SOURCE_WEIGHT = 0.5
ENGAGEMENT_WEIGHT = 0.5

## 6. MongoDB connection (for `06_load_mongodb`)

Same pattern as the storage connection string above: widget first, then 
environment variable, never a hardcoded default. `MONGO_URI` should 
already contain the target database name (e.g. 
`.../fomoless?retryWrites=true&w=majority`); `MONGO_COLLECTION_NAME` names 
the collection within it.

In [0]:
MONGO_URI = get_widget_or_env(
    "mongo_uri", "MONGODB_ATLAS_URI", "MongoDB Atlas Connection URI"
)
MONGO_DB_NAME = "fomoless"
MONGO_COLLECTION_NAME = "articles"

## 7. Helpers

In [0]:
def log(msg: str) -> None:
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts} UTC] {msg}")


def row_count(table_name: str) -> int:
    """Row count for a catalog table; 0 if it doesn't exist yet (e.g. gold on
    a first-ever run)."""
    try:
        return spark.table(table_name).count()
    except Exception:
        return 0


def run_stream(query, name: str) -> None:
    """Blocks until an `availableNow` streaming query drains its backlog,
    then logs a one-line summary."""
    query.awaitTermination()
    progress = query.lastProgress or {}
    rows_in = (progress.get("numInputRows") if progress else None)
    log(f"Stream '{name}' finished" + (f" — {rows_in} input row(s) this run" if rows_in is not None else "."))


def is_missing(value) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and pd.isna(value):
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def ensure_cdf_enabled(table_name: str) -> None:
    """Sets delta.enableChangeDataFeed on an EXISTING table if it isn't
    already on. Moved here from 03_silver_transform (this revision) since
    04_gold_aggregation now needs the identical logic for gold.articles --
    a no-op (one cheap SHOW TBLPROPERTIES) if it's already set. First-time
    table creation sets the property directly via .option(...) instead, in
    the notebook doing the creating."""
    if not spark.catalog.tableExists(table_name):
        return
    props = {
        row.key: row.value
        for row in spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()
    }
    if props.get("delta.enableChangeDataFeed", "").lower() != "true":
        spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
        log(f"delta.enableChangeDataFeed was missing on {table_name} -- enabled it now")


def checkpoint_exists(checkpoint_path: str) -> bool:
    """True if a streaming checkpoint already has recorded progress at this
    path -- i.e. this is NOT the first run of that stream.

    BUG FIX: both 04_gold_aggregation and 06_load_mongodb read their source
    table's Change Data Feed starting from version 0, assuming that's a
    harmless no-op once a checkpoint exists (true) AND a valid way to
    "snapshot" a table's full history on the very first run (false, for any
    table that already had data before CDF was turned on -- which is every
    table in this project, since CDF is new in this revision). Delta only
    keeps change records from the moment delta.enableChangeDataFeed was
    actually turned on; requesting an earlier startingVersion raises
    "change data was not recorded for version ..." instead of quietly
    covering the gap. This helper is what lets each notebook tell "first
    run, needs a one-time full backfill first" apart from "already running
    incrementally, checkpoint has the real position" -- see the backfill
    block right before each notebook's streaming query."""
    try:
        return len(dbutils.fs.ls(checkpoint_path)) > 0
    except Exception:
        return False


log(f"Config loaded — container: {CONTAINER_NAME}, catalog: {CATALOG_NAME}")